[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/03_lo_scout_migliora.ipynb)

# Lo scout migliora

Uno scout umano non giudica un calciatore guardando un solo numero. In questo episodio passiamo da una regressione con una variabile a una regressione con più variabili.

> **Missione** — Confrontare un modello semplice con un modello piu' informato, usando age, overall, potential e wage_eur.


Apriamo gli stessi dati. Oggi proveremo a dare allo scout più di un indizio per volta.


In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


Pulizia identica agli episodi precedenti.


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

Importiamo gli strumenti già noti: la regressione lineare e le tre metriche di errore (MAE, RMSE, $R^2$).


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.model_selection import train_test_split

> **Regressione multipla** — Nella regressione multipla il modello usa piu' input contemporaneamente. Geometricamente non stiamo piu' adattando una retta nel piano, ma un piano o un iperpiano in uno spazio con piu' dimensioni. L'idea resta la stessa: trovare la combinazione di variabili che riduce l'errore.


### Teoria — Regressione lineare multipla

Con $p$ variabili di input $x_1, x_2, \dots, x_p$, il modello diventa:

$\displaystyle \hat{y} \;=\; w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b$

Ogni variabile ha il suo peso $w_j$. In forma compatta, se $\mathbf{x} = (x_1, \dots, x_p)$ e $\mathbf{w} = (w_1, \dots, w_p)$:

$\displaystyle \hat{y} \;=\; \mathbf{w}^{\top}\mathbf{x} + b$

Geometricamente, con $1$ variabile cerchiamo una **retta**; con $2$ un **piano**; con $p$ un **iperpiano** in $p$ dimensioni. La logica resta la stessa: scegliere $\mathbf{w}$ e $b$ che minimizzano la stessa funzione di costo del notebook precedente:

$\displaystyle \mathrm{MSE}(\mathbf{w}, b) \;=\; \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$


## Modello A: solo overall

Per prima cosa ricostruiamo lo scout dell'episodio precedente, quello che usa solo l'overall. Sarà il nostro **punto di partenza**: il modello da battere.


In [ ]:
X_one = df[["overall"]]
y = df["value_mln_eur"]

model_one = LinearRegression()
model_one.fit(X_one, y)
pred_one = model_one.predict(X_one)

mae_one = mean_absolute_error(y, pred_one)
rmse_one = mean_squared_error(y, pred_one) ** 0.5
r2_one = r2_score(y, pred_one)

print(f"MAE modello A: {mae_one:.2f} milioni")
print(f"RMSE modello A: {rmse_one:.2f} milioni")
print(f"R^2 modello A: {r2_one:.3f}")

## Modello B: più indizi

Adesso lasciamo allo scout più informazioni — età, overall, potential e salario — e vediamo se l'errore diminuisce. Notate che il codice è quasi identico a prima: cambia solo la lista delle colonne di input.


In [ ]:
features = [
    c for c in ["age", "overall", "potential", "wage_eur"]
    if c in df.columns
]
X_many = df[features]

model_many = LinearRegression()
model_many.fit(X_many, y)
pred_many = model_many.predict(X_many)

mae_many = mean_absolute_error(y, pred_many)
rmse_many = mean_squared_error(y, pred_many) ** 0.5
r2_many = r2_score(y, pred_many)

print("Feature usate:", features)
print(f"MAE modello B: {mae_many:.2f} milioni")
print(f"RMSE modello B: {rmse_many:.2f} milioni")
print(f"R^2 modello B: {r2_many:.3f}")

> **Confronto tra modelli** — Un modello piu' ricco puo' usare piu' informazione. Se aggiungiamo variabili sensate, l'errore tende a diminuire. Pero' aggiungere variabili non e' sempre una vittoria: alcune colonne possono essere rumorose, ridondanti o persino fuorvianti.


## Confrontiamo gli errori

Mettiamo i due modelli faccia a faccia in una tabella: stessi giocatori, stesso target, due *ricette* diverse. Chi vince?


In [ ]:
comparison = pd.DataFrame({
    "modello": ["A: solo overall", "B: piu' indizi"],
    "MAE_mln": [mae_one, mae_many],
    "RMSE_mln": [rmse_one, rmse_many],
    "R2": [r2_one, r2_many]
})
comparison.round(3)

## Quanto pesa ogni variabile?

Una domanda naturale: nella nuova formula, quale indizio conta di più? Stampiamo i coefficienti del modello B, uno per ogni variabile.


In [ ]:
coef = pd.DataFrame({
    "feature": features,
    "coefficiente": model_many.coef_,
})
coef.sort_values("coefficiente", ascending=False)

> **Attenzione ai coefficienti** — I coefficienti non sono sempre confrontabili direttamente, perche' le variabili hanno scale diverse. Un punto di overall e un euro di salario non hanno la stessa unita'. Per interpretazioni serie bisognerebbe normalizzare le variabili. Qui usiamo i coefficienti solo per intuire che il modello combina piu' indizi.


## Confronto onesto dei pesi: standardizzare le variabili


### Teoria — Standardizzare per confrontare i pesi

Le nostre variabili hanno **scale molto diverse**: l'overall va da circa $50$ a $95$, il salario da migliaia a centinaia di migliaia di euro. Il coefficiente del salario è numericamente piccolo solo perché il salario è un numero grande, non perché il salario "conti poco".

Per rendere i pesi confrontabili, trasformiamo ogni variabile così:

$\displaystyle z_j \;=\; \frac{x_j - \bar{x}_j}{\sigma_j}$

dove $\bar{x}_j$ è la media e $\sigma_j$ la deviazione standard della $j$-esima variabile. Dopo la trasformazione, tutte le variabili hanno **media $0$ e deviazione standard $1$**: ora il coefficiente più grande in valore assoluto indica la variabile più influente.


Applichiamo la trasformazione: ora ogni variabile ha media $0$ e deviazione standard $1$. Riaddestriamo il modello sui dati standardizzati e leggiamo i nuovi pesi, finalmente confrontabili tra loro.


In [ ]:
from sklearn.preprocessing import StandardScaler

# Standardizziamo le feature: media 0, deviazione standard 1 per ognuna
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_many)

model_scaled = LinearRegression()
model_scaled.fit(X_scaled, y)

# Ora i pesi sono confrontabili
pesi = (
    pd.DataFrame({
        "feature": features,
        "peso_standardizzato": model_scaled.coef_,
    })
    .sort_values(
        "peso_standardizzato",
        key=lambda s: s.abs(),
        ascending=False,
    )
)
pesi.round(3)


Visualizziamo i pesi standardizzati con un grafico a barre. Più una barra è lunga in valore assoluto, più quella variabile è influente. Le barre verdi sono pesi positivi (al crescere della variabile cresce il valore), quelle rosse sono negative.


In [ ]:
# Visualizziamo i pesi standardizzati
plt.figure(figsize=(7,4))
colori = [
    "#2a9d8f" if w > 0 else "#e76f51"
    for w in pesi["peso_standardizzato"]
]
plt.barh(pesi["feature"], pesi["peso_standardizzato"], color=colori)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Peso (su variabili standardizzate)")
plt.title("Quale variabile pesa di piu' sul valore di mercato?")
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis="x")
plt.show()


## Lo scout automatico su giocatori casuali

Test finale dell'episodio: prendiamo 12 giocatori a caso e per ognuno mostriamo cosa prevede il modello *solo overall* e cosa prevede il modello *più indizi*. A occhio, quanto migliora il secondo?


In [ ]:
sample = df.sample(12, random_state=13).copy()
sample["pred_solo_overall"] = model_one.predict(sample[["overall"]])
sample["pred_piu_indizi"] = model_many.predict(sample[features])
sample[[
    "short_name", "age", "overall", "potential", "wage_eur",
    "value_mln_eur", "pred_solo_overall", "pred_piu_indizi"
]].round(2)

---

> **Domanda** — Trovate un giocatore per cui il modello con piu' indizi migliora molto rispetto al modello con solo overall. Quale informazione extra probabilmente lo ha aiutato?


---

> **Cosa dovremmo aver capito** — La regressione non e' solo una retta: e' un modo per costruire una funzione che trasforma informazioni in una previsione numerica. Nel prossimo episodio faremo il controllo piu' importante: il modello funziona anche su giocatori che non ha usato per imparare?
